<a href="https://colab.research.google.com/github/ibnu-aulia/arabic-text-analyzer/blob/main/arabic_review_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic Restaurant Review Sentiment Classifier

Binary sentiment classification (positive / negative) on Arabic restaurant reviews, using TF-IDF features and scikit-learn.

**Dataset:** ArRestReviews (`hadyelsahar/ar_res_reviews`, Hugging Face), 8,364 reviews.
**Focus:** Arabic text normalization, handling class imbalance, and understanding why the model fails.

## 1. Setup and load data

In [1]:
import re
from collections import Counter

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

data = load_dataset("hadyelsahar/ar_res_reviews")
df = data["train"].to_pandas()

print(df.shape)
print(df.head())
print(df["polarity"].value_counts())

README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.89MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8364 [00:00<?, ? examples/s]

(8364, 4)
   polarity                                               text restaurant_id  \
0         0  اولا: المنيو تغير الشورما اصبحت اعتياديه بأختف...           296   
1         0  من محلات الشاورما ذات الشعبيه لتميز الصلصات ال...           296   
2         1  دجاج طازج يحضر امامك على الطلب لا يقوم باعدة ا...          5027   
3         1  فكما تعرف أستراليا بالكنغر والكوالا. فإنها تعر...           642   
4         0  إسمحو لي أن أقيم مطعم هاشم بصفتي فلسطيني عشت ف...           434   

  user_id  
0     423  
1     423  
2   39580  
3     444  
4    2191  
polarity
1    5946
0    2418
Name: count, dtype: int64


## 2. Exploratory data analysis

Labels are imbalanced (about 71% positive), negative reviews are longer on average, and the most frequent words are mostly function words and punctuation.

In [2]:
# Review length in words
df["n_words"] = df["text"].str.split().str.len()
print(df["n_words"].describe())
print(df.groupby("polarity")["n_words"].mean())

# Exact duplicate texts
print("Duplicate texts:", df["text"].duplicated().sum())

# Most frequent words (raw text)
words = " ".join(df["text"]).split()
print(Counter(words).most_common(10))

count    8364.000000
mean       40.299139
std        51.937850
min         1.000000
25%        11.000000
50%        24.000000
75%        48.000000
max       800.000000
Name: n_words, dtype: float64
polarity
0    45.205542
1    38.303902
Name: n_words, dtype: float64
Duplicate texts: 64
[('من', 6976), ('في', 5163), ('و', 3501), ('المطعم', 3373), ('..', 3143), ('ما', 2504), ('على', 2480), ('جدا', 1874), ('مع', 1768), ('بس', 1646)]


## 3. Arabic text normalization

Unify alef variants (أ إ آ → ا), alef maqsurah (ى → ي), and remove everything that is not an Arabic letter (punctuation, digits, Latin letters, emoji).

In [3]:
def normalize_arabic(text):
    text = text.replace("أ", "ا")
    text = text.replace("إ", "ا")
    text = text.replace("آ", "ا")
    text = text.replace("ى", "ي")
    text = re.sub(r"[^\u0621-\u064A\s]", " ", text)
    text = " ".join(text.split())
    return text

print(normalize_arabic("إسمحو لي أن أقيم المطعم.."))

df["clean_text"] = df["text"].apply(normalize_arabic)
print(df[["text", "clean_text"]].head(3))

# Most frequent words after cleaning
words_clean = " ".join(df["clean_text"]).split()
print(Counter(words_clean).most_common(10))

اسمحو لي ان اقيم المطعم
                                                text  \
0  اولا: المنيو تغير الشورما اصبحت اعتياديه بأختف...   
1  من محلات الشاورما ذات الشعبيه لتميز الصلصات ال...   
2  دجاج طازج يحضر امامك على الطلب لا يقوم باعدة ا...   

                                          clean_text  
0  اولا المنيو تغير الشورما اصبحت اعتياديه باختفا...  
1  من محلات الشاورما ذات الشعبيه لتميز الصلصات ال...  
2  دجاج طازج يحضر امامك علي الطلب لا يقوم باعدة ا...  
[('من', 7053), ('في', 5268), ('و', 3579), ('المطعم', 3527), ('جدا', 3259), ('علي', 2658), ('ما', 2551), ('مع', 1792), ('ان', 1752), ('بس', 1692)]


### Remove empty and duplicate reviews

47 reviews became empty after cleaning (emoji, links, English, Korean characters, Arabic written in Latin letters). Duplicates are removed so the same review cannot appear in both train and test sets.

In [4]:
print("Empty after cleaning:", (df["clean_text"].str.len() == 0).sum())
df = df[df["clean_text"].str.len() > 0]
df = df.reset_index(drop=True)

print("Duplicates after cleaning:", df["clean_text"].duplicated().sum())
df = df.drop_duplicates(subset="clean_text")
df = df.reset_index(drop=True)

print(df.shape)
print(df["polarity"].value_counts())

Empty after cleaning: 47
Duplicates after cleaning: 80
(8237, 6)
polarity
1    5850
0    2387
Name: count, dtype: int64


## 4. Train/test split and baseline

80% train / 20% test, stratified so the 71/29 label ratio is the same in both. A "lazy" model that always predicts positive gives the baseline to beat.

In [5]:
X = df["clean_text"]
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(len(X_train), len(X_test))
print(y_train.value_counts())
print(y_test.value_counts())

print("Baseline (always positive):", round((y_test == 1).mean(), 4))

6589 1648
polarity
1    4680
0    1909
Name: count, dtype: int64
polarity
1    1170
0     478
Name: count, dtype: int64
Baseline (always positive): 0.71


## 5. Features: TF-IDF

The vectorizer learns its vocabulary from the training set only. The test set is only transformed, never fitted.

In [6]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape, X_test_vec.shape)

(6589, 5000) (1648, 5000)


## 6. Model 1: Logistic Regression

In [7]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)
predictions = model.predict(X_test_vec)

print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.8379854368932039
              precision    recall  f1-score   support

           0       0.87      0.52      0.65       478
           1       0.83      0.97      0.89      1170

    accuracy                           0.84      1648
   macro avg       0.85      0.74      0.77      1648
weighted avg       0.84      0.84      0.82      1648



Accuracy looks good, but recall on the negative class is low (0.52): the model is biased toward the majority class.

## 7. Model 2: Logistic Regression with balanced class weights

In [8]:
model2 = LogisticRegression(max_iter=1000, class_weight="balanced")
model2.fit(X_train_vec, y_train)
predictions2 = model2.predict(X_test_vec)

print(classification_report(y_test, predictions2))

              precision    recall  f1-score   support

           0       0.71      0.76      0.74       478
           1       0.90      0.87      0.89      1170

    accuracy                           0.84      1648
   macro avg       0.81      0.82      0.81      1648
weighted avg       0.85      0.84      0.84      1648



## 8. Model 3: Random Forest (comparison)

In [9]:
model3 = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
model3.fit(X_train_vec, y_train)
predictions3 = model3.predict(X_test_vec)

print(classification_report(y_test, predictions3))

              precision    recall  f1-score   support

           0       0.83      0.53      0.65       478
           1       0.83      0.95      0.89      1170

    accuracy                           0.83      1648
   macro avg       0.83      0.74      0.77      1648
weighted avg       0.83      0.83      0.82      1648



A more complex model did not help: Random Forest behaves like Model 1 on the negative class.

## 9. Experiment: bigrams (word pairs)

Idea: negation such as “ما انصح” (I do not recommend) might be captured by word pairs.

In [10]:
vectorizer2 = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec2 = vectorizer2.fit_transform(X_train)
X_test_vec2 = vectorizer2.transform(X_test)

model4 = LogisticRegression(max_iter=1000, class_weight="balanced")
model4.fit(X_train_vec2, y_train)
predictions4 = model4.predict(X_test_vec2)

print(classification_report(y_test, predictions4))

              precision    recall  f1-score   support

           0       0.71      0.77      0.74       478
           1       0.90      0.87      0.89      1170

    accuracy                           0.84      1648
   macro avg       0.81      0.82      0.81      1648
weighted avg       0.85      0.84      0.84      1648



### Experiment: bigrams with more features (20,000)

In [11]:
vectorizer3 = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_vec3 = vectorizer3.fit_transform(X_train)
X_test_vec3 = vectorizer3.transform(X_test)

model5 = LogisticRegression(max_iter=1000, class_weight="balanced")
model5.fit(X_train_vec3, y_train)
predictions5 = model5.predict(X_test_vec3)

print(classification_report(y_test, predictions5))

              precision    recall  f1-score   support

           0       0.74      0.77      0.76       478
           1       0.91      0.89      0.90      1170

    accuracy                           0.86      1648
   macro avg       0.82      0.83      0.83      1648
weighted avg       0.86      0.86      0.86      1648



## 10. Results

| | Features | Accuracy | Recall (neg) | Precision (neg) | F1 (neg) | Macro F1 |
|---|---|---|---|---|---|---|
| Baseline (always positive) | - | 0.71 | 0.00 | - | - | - |
| 1. Logistic Regression | 5k unigrams | 0.84 | 0.52 | 0.87 | 0.65 | 0.77 |
| 2. LR, balanced | 5k unigrams | 0.84 | 0.76 | 0.71 | 0.74 | 0.81 |
| 3. Random Forest, balanced | 5k unigrams | 0.83 | 0.53 | 0.83 | 0.65 | 0.77 |
| 4. LR, balanced | 5k uni+bigrams | 0.84 | 0.77 | 0.71 | 0.74 | 0.81 |
| 5. LR, balanced | 20k uni+bigrams | 0.86 | 0.77 | 0.74 | 0.76 | 0.83 |

Balanced class weights were the biggest improvement: they raised negative recall from 0.52 to 0.76 at the cost of some precision. Bigrams alone changed almost nothing; a larger feature space improved precision slightly.

## 11. Error analysis: negative reviews the model missed

Reading the mistakes of Model 2 (113 negative reviews predicted as positive).

In [12]:
errors = X_test[(y_test == 0) & (predictions2 == 1)]
print(len(errors))
print(errors.head(5).tolist())

113
['اسعاره رخيصه والخدمه منوسطه والنظافه ممتازه الطعم عادي خاصه باالاكل الهندي فيه اللي افضل منه', 'المكان حلووو بس الطعم مفقوود ما انصح فيه', 'هو اكلهم حلو وماعليه بس مافيه نظافة بالاكل مره حصلت حشرات بالاكل', 'خصوصا الباربيكيو بيتزا', 'لايعجبني المطعم ا لذي به معسلات هذا المطعم اشبه بالقهوة الشعبية وهذا المطعم غير محترم']


Main patterns found by reading the errors:

1. **Negation** (لا، ما، غير، مافيه): “ما انصح” and “غير محترم” are read as positive because TF-IDF counts words, not their order.
2. **Informal spelling and dialect**: stretched letters (حلووو), split words, ه/ة endings.
3. **Mixed or noisy labels**: some reviews mention both good and bad points, and some are fragments with no sentiment words.

## 12. Try the model on new reviews

In [13]:
new_reviews = [
    "الأكل رائع جدا والخدمة ممتازة",
    "المطعم سيء جدا والطعام بارد",
    "الخدمة بطيئة لكن الطعم لذيذ",
]

cleaned = [normalize_arabic(t) for t in new_reviews]
new_vec = vectorizer.transform(cleaned)

print(model2.predict(new_vec))
print(model2.predict_proba(new_vec))

[1 0 1]
[[0.02678391 0.97321609]
 [0.93783625 0.06216375]
 [0.27695551 0.72304449]]


The model is confident on clear reviews (97% and 94%) and less sure on the mixed one (72%).

## Conclusion

The best model (TF-IDF with unigrams and bigrams, 20,000 features, balanced class weights) reached 0.86 accuracy and 0.76 F1 on the negative class. A model that always predicts “positive” scores 0.71, so the model does learn from the text.

## Limitations

- Negation is the main failure. Bigrams did not fix it on about 6.5k training reviews.
- Informal spelling and dialect are not fully handled.
- Arabizi (Arabic written in Latin letters) is dropped by the cleaning step.
- Five models were compared on the same test set, so the best score is slightly optimistic. Cross-validation would be a cleaner check.

## Next steps

- Cross-validation and a separate validation set.
- Handle negation and dialect in preprocessing.
- Fine-tune a pretrained Arabic model (e.g. AraBERT) and compare.